# UBL price-only MLP — single split and walk-forward

Price-only features. 
Model A predicts the future close; Model B predicts the log return and converts it to price.

## A. Data preparation, features, windows, and single-split baseline

In [ ]:
# A-1. IMPORTS
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential, layers, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error as MAE, mean_squared_error as MSE, r2_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
# A-2. LOAD AND CLEAN PRICE DATA
# Load the same UBL price dataset used by the source notebook.
price_path = "/kaggle/input/datasets/ba2lll/ubl-price/UBL.csv"
if not os.path.exists(price_path):
    candidates = glob.glob("/kaggle/input/**/UBL.csv", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find UBL.csv under /kaggle/input.")
    price_path = candidates[0]
df_raw = pd.read_csv(price_path)
df_raw.columns = df_raw.columns.str.strip().str.upper()
df_raw["DATE"] = pd.to_datetime(df_raw["DATE"])
df_raw = df_raw[df_raw["DATE"] >= "2021-01-01"].drop_duplicates()
ok = ((df_raw[["OPEN","HIGH","LOW","CLOSE"]] > 0).all(axis=1) &
      (df_raw["VOLUME"] >= 0) &
      (df_raw["HIGH"] >= df_raw[["OPEN","CLOSE","LOW"]].max(axis=1)) &
      (df_raw["LOW"] <= df_raw[["OPEN","CLOSE","HIGH"]].min(axis=1)))
df = df_raw.loc[ok].sort_values("DATE").copy()
df = df[(df.DATE >= "2021-09-30") & (df.DATE <= "2026-06-30")].reset_index(drop=True)
assert df.DATE.is_unique, "Duplicate dates remain after cleaning."
print("Cleaned rows:", len(df), "| range:", df.DATE.min().date(), "to", df.DATE.max().date())
assert len(df) == 1176, f"Expected 1,176 cleaned rows; found {len(df)}. Check dataset/version and cleaning."

In [ ]:
# A-3. SEVEN PRICE FEATURES
# Seven source-notebook features; do not add or remove features.
df["ret"] = np.log(df.CLOSE).diff()
df["hl_range"] = (df.HIGH - df.LOW) / df.CLOSE
df["oc_change"] = (df.CLOSE - df.OPEN) / df.OPEN
df["log_vol"] = np.log1p(df.VOLUME)
df["ma5_ratio"] = df.CLOSE / df.CLOSE.rolling(5).mean() - 1
df["ma20_ratio"] = df.CLOSE / df.CLOSE.rolling(20).mean() - 1
df["vol10"] = df.ret.rolling(10).std()
df = df.dropna().reset_index(drop=True)
feats = ["ret","hl_range","oc_change","log_vol","ma5_ratio","ma20_ratio","vol10"]
W = 30
assert len(feats) == 7
print("Feature rows after rolling-feature warmup:", len(df))

In [ ]:
# A-4. 30×7 WINDOWS AND TARGETS
# Window ending at t predicts target date t+1. Keep the source notebook's 30x7 window.
X, y_return, y_price, current_close, target_dates = [], [], [], [], []
for i in range(W-1, len(df)):
    X.append(df[feats].iloc[i-W+1:i+1].to_numpy(dtype=float))
    y_return.append(np.log(df.CLOSE.iloc[i+1] / df.CLOSE.iloc[i]) if i+1 < len(df) else np.nan)
    y_price.append(df.CLOSE.iloc[i+1] if i+1 < len(df) else np.nan)
    current_close.append(df.CLOSE.iloc[i])
    target_dates.append(df.DATE.iloc[i+1] if i+1 < len(df) else pd.NaT)
# Use full raw cleaned price series to retain the final target observation.
# df's last row has no next-day target; source notebook's shift(-1) drops it.
df_feat = df.copy()
df_feat["target_ret"] = np.log(df_feat.CLOSE.shift(-1) / df_feat.CLOSE)
df_feat["target_close"] = df_feat.CLOSE.shift(-1)
df_feat["tdate"] = df_feat.DATE.shift(-1)
df_feat = df_feat.dropna(subset=feats + ["target_ret","target_close","tdate"]).reset_index(drop=True)
X_list, yr, yp, pc, td = [], [], [], [], []
for i in range(W-1, len(df_feat)):
    X_list.append(df_feat[feats].iloc[i-W+1:i+1].to_numpy(dtype=float))
    yr.append(df_feat.target_ret.iloc[i]); yp.append(df_feat.target_close.iloc[i])
    pc.append(df_feat.CLOSE.iloc[i]); td.append(df_feat.tdate.iloc[i])
X = np.asarray(X_list); y_return = np.asarray(yr); y_price = np.asarray(yp)
current_close = np.asarray(pc); target_dates = pd.to_datetime(td)
print("Window tensor:", X.shape, "| flattened inputs:", W*len(feats))
assert X.shape[1:] == (30,7) and X.reshape(len(X),-1).shape[1] == 210

In [ ]:
# A-5. CHRONOLOGICAL SPLIT AND FEATURE SCALING
# Chronological split boundaries are target dates, exactly as in the source notebook.
tr = np.asarray(target_dates <= pd.Timestamp("2025-01-23"))
va = np.asarray((target_dates > pd.Timestamp("2025-01-23")) & (target_dates <= pd.Timestamp("2025-10-10")))
te = np.asarray(target_dates > pd.Timestamp("2025-10-10"))
print("Train / validation / test:", tr.sum(), va.sum(), te.sum())
assert te.sum() == 177, f"Expected h=1 test rows=177; found {te.sum()}."
X_flat = X.reshape(len(X), -1)
sx = StandardScaler().fit(X_flat[tr])
X_scaled = sx.transform(X_flat)
X_train, X_val, X_test = X_scaled[tr], X_scaled[va], X_scaled[te]

In [ ]:
# A-6. SINGLE-SPLIT MLP MODELS
def build_mlp(n_features):
    tf.keras.utils.set_random_seed(SEED)
    model = Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def fit_single(target, target_mask_train=tr):
    # X scaler and target scaler are fitted on training rows only.
    sy = StandardScaler().fit(target[target_mask_train].reshape(-1,1))
    model = build_mlp(X_train.shape[1])
    model.fit(X_train, sy.transform(target[tr].reshape(-1,1)),
              validation_data=(X_val, sy.transform(target[va].reshape(-1,1))),
              epochs=200, batch_size=32, verbose=0,
              callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
    return model, sy

model_A, sy_A = fit_single(y_price)
model_B, sy_B = fit_single(y_return)
pred_A = model_A.predict(X_test, verbose=0).ravel()
pred_A = sy_A.inverse_transform(pred_A.reshape(-1,1)).ravel()
pred_B_ret = model_B.predict(X_test, verbose=0).ravel()
pred_B_ret = sy_B.inverse_transform(pred_B_ret.reshape(-1,1)).ravel()
pred_B = current_close[te] * np.exp(pred_B_ret)

### Single-split metrics

In [ ]:
# A-7. SINGLE-SPLIT METRICS
def metrics(actual, pred, prev, naive=False):
    actual, pred, prev = map(np.asarray, (actual, pred, prev))
    return {
        "MAE": MAE(actual, pred),
        "RMSE": np.sqrt(MSE(actual, pred)),
        "MAPE": np.mean(np.abs((actual-pred)/actual))*100,
        "R2": r2_score(actual, pred),
        "DirAcc": np.nan if naive else np.mean(np.sign(actual-prev) == np.sign(pred-prev))*100
    }

single_rows = []
for split_name, mask in [("Train",tr),("Validation",va),("Test",te)]:
    # Predict each split for the single-split model; target scalers are training-only.
    xa = X_scaled[mask]
    pa = sy_A.inverse_transform(model_A.predict(xa, verbose=0)).ravel()
    pb_ret = sy_B.inverse_transform(model_B.predict(xa, verbose=0)).ravel()
    pb = current_close[mask] * np.exp(pb_ret)
    actual, prev = y_price[mask], current_close[mask]
    for name, pred, naive in [("Model A (price)",pa,False),("Model B (return)",pb,False),("Naive",prev,True)]:
        single_rows.append({"Split":split_name,"Model":name,**metrics(actual,pred,prev,naive=naive)})
single_summary = pd.DataFrame(single_rows)
display(single_summary.round(3))
single_summary.to_csv("ubl_price_single_split_metrics.csv", index=False)

## B. Walk-forward validation

In [ ]:
# WF-1. CONFIG
WF_STEP = 21
WF_INNER_VAL = 0.15
WF_START = int(np.flatnonzero(te)[0])
SEED = 42
print("WF start position:", WF_START, "| test target date:", target_dates[WF_START].date())

In [ ]:
# WF-2. MODEL BUILDER
def build_mlp_wf(n_features):
    tf.keras.utils.set_random_seed(SEED)
    model = Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def fit_predict_fold(train_pool, test_pos, h=1):
    # Purge training targets that reach or cross the fold start; then keep an h-row gap
    # between fit rows and inner-validation rows. Both scalers fit on fit rows only.
    eligible = train_pool[train_pool + h <= test_pos[0]]
    if len(eligible) < 20:
        raise ValueError(f"Too few eligible training rows ({len(eligible)}) for fold.")
    n_val = max(1, int(np.ceil(len(eligible) * WF_INNER_VAL)))
    cut = len(eligible) - n_val - h
    if cut < 1:
        raise ValueError("Not enough rows after applying the h-day inner-validation gap.")
    fit_pos = eligible[:cut]
    val_pos = eligible[cut+h:]
    if len(val_pos) == 0:
        raise ValueError("Inner validation is empty after purge gap.")
    sx_fold = StandardScaler().fit(X_flat[fit_pos])
    Xf = sx_fold.transform(X_flat[fit_pos])
    Xv = sx_fold.transform(X_flat[val_pos])
    Xt = sx_fold.transform(X_flat[test_pos])
    out = {}
    for tag, target in [("A", y_price), ("B", y_return)]:
        sy = StandardScaler().fit(target[fit_pos].reshape(-1,1))
        yf = sy.transform(target[fit_pos].reshape(-1,1))
        yv = sy.transform(target[val_pos].reshape(-1,1))
        model = build_mlp_wf(Xf.shape[1])
        model.fit(Xf, yf, validation_data=(Xv,yv), epochs=200, batch_size=32, verbose=0,
                  callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)])
        out[tag] = sy.inverse_transform(model.predict(Xt, verbose=0)).ravel()
    cur = current_close[test_pos]
    return out["A"], cur * np.exp(out["B"]), cur, eligible, fit_pos, val_pos

In [ ]:
# WF-3. EXPANDING AND ROLLING LOOP
def walk_forward(mode="expanding"):
    rows = []
    fold_starts = list(range(WF_START, len(X), WF_STEP))
    first_eligible = np.arange(0, fold_starts[0])
    first_eligible = first_eligible[first_eligible + 1 <= fold_starts[0]]
    roll_window = len(first_eligible)
    if mode == "rolling":
        print("ROLL_WINDOW (eligible training rows at fold 1):", roll_window)
    for fold, start in enumerate(fold_starts, start=1):
        end = min(start + WF_STEP, len(X))
        test_pos = np.arange(start, end)
        pool_lo = 0 if mode == "expanding" else max(0, start-roll_window)
        pool = np.arange(pool_lo, start)
        pA, pB, cur, eligible, fit_pos, val_pos = fit_predict_fold(pool, test_pos, h=1)
        train_end_date = target_dates[eligible[-1]]
        test_start_date = target_dates[test_pos[0]]
        print(f"[{mode}] fold {fold:>2} | eligible train={len(eligible):>4} "
              f"({target_dates[eligible[0]].date()} -> {train_end_date.date()}) | "
              f"test={len(test_pos):>2} ({test_start_date.date()} -> {target_dates[test_pos[-1]].date()}) | "
              f"train end < test start: {train_end_date < test_start_date}")
        assert train_end_date < test_start_date, "Temporal leakage: train end is not before test start."
        rows.append(pd.DataFrame({
            "fold":fold, "DATE":target_dates[test_pos], "prev_close":cur,
            "actual":y_price[test_pos], "pred_A":pA, "pred_B":pB,
            "naive":cur, "train_size":len(eligible), "fit_size":len(fit_pos),
            "val_size":len(val_pos), "train_start":target_dates[eligible[0]],
            "train_end":train_end_date, "test_start":test_start_date
        }))
    result = pd.concat(rows, ignore_index=True)
    assert result.fold.nunique() == 9, f"Expected 9 h=1 folds; found {result.fold.nunique()}."
    return result, roll_window

wf_expanding, _ = walk_forward("expanding")
wf_rolling, ROLL_WINDOW = walk_forward("rolling")
print("ROLL_WINDOW =", ROLL_WINDOW)

In [ ]:
# WF-4. SUMMARY TABLE
def summarize_wf(wf, label):
    rows = []
    for name, col, is_naive in [("Model A","pred_A",False),("Model B","pred_B",False),("Naive","naive",True)]:
        rows.append({"Evaluation":label,"Model":name,**metrics(wf.actual, wf[col], wf.prev_close, naive=is_naive)})
    return pd.DataFrame(rows)

single_test = single_summary[single_summary.Split=="Test"].copy()
single_test = single_test.rename(columns={"Model":"Model"})
single_test.insert(0,"Evaluation","Single split")
summary_wf = pd.concat([single_test[["Evaluation","Model","MAE","RMSE","MAPE","R2","DirAcc"]],
                        summarize_wf(wf_expanding,"WF expanding"),
                        summarize_wf(wf_rolling,"WF rolling")], ignore_index=True)
always_up = (wf_expanding.actual > wf_expanding.prev_close).mean()*100
print(f"Walk-forward test rows: {len(wf_expanding)} | always-up rate: {always_up:.1f}%")
display(summary_wf.round(3))
summary_wf.to_csv("ubl_price_walkforward_summary.csv", index=False)

In [ ]:
# WF-5. PER-FOLD STABILITY
def per_fold(wf):
    out=[]
    for fold,g in wf.groupby("fold"):
        out.append({
            "fold":fold, "test_from":pd.to_datetime(g.DATE.iloc[0]).date(), "n":len(g),
            "A_MAE":MAE(g.actual,g.pred_A), "A_Dir":metrics(g.actual,g.pred_A,g.prev_close)["DirAcc"],
            "B_MAE":MAE(g.actual,g.pred_B), "B_Dir":metrics(g.actual,g.pred_B,g.prev_close)["DirAcc"],
            "Naive_MAE":MAE(g.actual,g.naive)
        })
    return pd.DataFrame(out)
fold_exp = per_fold(wf_expanding); fold_rol = per_fold(wf_rolling)
print("EXPANDING WINDOW"); display(fold_exp.round(3))
print("ROLLING WINDOW"); display(fold_rol.round(3))
print("Expanding mean/std across folds"); display(fold_exp[["A_MAE","B_MAE","Naive_MAE","A_Dir","B_Dir"]].agg(["mean","std"]).round(3))
print("Rolling mean/std across folds"); display(fold_rol[["A_MAE","B_MAE","Naive_MAE","A_Dir","B_Dir"]].agg(["mean","std"]).round(3))
print("Folds where Model B beats naive (expanding):", int((fold_exp.B_MAE < fold_exp.Naive_MAE).sum()), "/", len(fold_exp))
print("Folds where Model B beats naive (rolling):", int((fold_rol.B_MAE < fold_rol.Naive_MAE).sum()), "/", len(fold_rol))
fold_exp.to_csv("ubl_price_walkforward_expanding_per_fold.csv", index=False)
fold_rol.to_csv("ubl_price_walkforward_rolling_per_fold.csv", index=False)

In [ ]:
# WF-6. PLOTS AND PREDICTION EXPORTS
fig, axes = plt.subplots(2,1,figsize=(13,9),sharex=True)
for ax,wf,title in [(axes[0],wf_expanding,"Expanding window"),(axes[1],wf_rolling,"Rolling window")]:
    ax.plot(wf.DATE,wf.actual,label="Actual",color="black",lw=1.5)
    ax.plot(wf.DATE,wf.pred_A,label="Model A (price target)",alpha=.85)
    ax.plot(wf.DATE,wf.pred_B,label="Model B (return target)",alpha=.85)
    for start_date in wf.groupby("fold").DATE.first().iloc[1:]:
        ax.axvline(pd.to_datetime(start_date),color="grey",ls=":",lw=.8)
    ax.set_title(f"{title} (dotted lines = retraining points)")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("ubl_price_walkforward_predictions.png",dpi=200); plt.show()

fig,ax=plt.subplots(figsize=(11,4))
x=np.arange(len(fold_exp)); width=.27
ax.bar(x-width,fold_exp.A_MAE,width,label="Model A")
ax.bar(x,fold_exp.B_MAE,width,label="Model B")
ax.bar(x+width,fold_exp.Naive_MAE,width,label="Naive")
ax.set_xticks(x); ax.set_xticklabels(fold_exp.fold)
ax.set_xlabel("Fold"); ax.set_ylabel("MAE (PKR)"); ax.set_title("Per-fold MAE — expanding window")
ax.legend(); plt.tight_layout(); plt.savefig("ubl_price_walkforward_fold_mae.png",dpi=200); plt.show()
wf_expanding.to_csv("ubl_price_walkforward_expanding_predictions.csv",index=False)
wf_rolling.to_csv("ubl_price_walkforward_rolling_predictions.csv",index=False)